In [9]:
import warnings
warnings.filterwarnings("ignore")

## RQ1 - RepGen v Baseline Models

In [10]:
from typing import Dict, Any, List
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

# --- Configuration --------------------------------------------------------
FILEPATH = "ICSE26-Code-Gen-Results - McNemar.csv"
PRIMARY_MODEL_TUPLE = ('RepGen', 'Baseline')

# --- Statistical Function ---------------------------------------------------
def calculate_mcnemar_stats(repgen_col: pd.Series, baseline_col: pd.Series) -> Dict[str, Any]:
    repgen_col = repgen_col.astype(bool)
    baseline_col = baseline_col.astype(bool)

    a = int(np.sum(repgen_col & baseline_col))  # Both Y
    b = int(np.sum(repgen_col & ~baseline_col)) # RepGen Y / Baseline N
    c = int(np.sum(~repgen_col & baseline_col)) # RepGen N / Baseline Y
    d = int(np.sum(~repgen_col & ~baseline_col))# Both N
    n = a + b + c + d

    if n == 0:
        return {k: np.nan for k in [
            'a','b','c','d','p_value',
            'RepGen_Prop','RepGen_CI_Low','RepGen_CI_High',
            'Baseline_Prop','Baseline_CI_Low','Baseline_CI_High',
            'Prop_Diff','Prop_Diff_CI_Low','Prop_Diff_CI_High',
            'Odds_Ratio'
        ]}

    table = [[a, b], [c, d]]

    # 1. McNemar's exact p-value
    p_value = 1.0 if (b + c == 0) else float(sm.stats.mcnemar(table, exact=True).pvalue)

    # 2. Individual proportions (Wilson CI)
    repgen_successes = a + b
    baseline_successes = a + c
    p_repgen = repgen_successes / n
    p_baseline = baseline_successes / n
    ci_repgen = proportion_confint(repgen_successes, n, method='wilson')
    ci_baseline = proportion_confint(baseline_successes, n, method='wilson')

    # 3. Paired proportion difference (Newcombe hybrid score CI)
    prop_diff = (b - c) / n
    if b + c == 0:
        prop_diff_ci_low, prop_diff_ci_high = 0.0, 0.0
    else:
        ci_theta = proportion_confint(b, b + c, method='wilson')
        prop_diff_ci_low = (2 * ci_theta[0] - 1) * (b + c) / n
        prop_diff_ci_high = (2 * ci_theta[1] - 1) * (b + c) / n

    # 4. Haldane–Anscombe odds ratio
    odds_ratio = (b + 0.5) / (c + 0.5)

    return {
        'Both_Reproduced (a)': a,
        'Unique_Bugs_Primary (b)': b,
        'Baseline (c)': c,
        'None_Reproduced (d)': d,
        'p_value': p_value,
        'Prop_Diff': prop_diff,
        'Prop_Diff_CI_Low': prop_diff_ci_low,
        'Prop_Diff_CI_High': prop_diff_ci_high,
        'RepGen_Prop': p_repgen,
        'RepGen_CI_Low': ci_repgen[0],
        'RepGen_CI_High': ci_repgen[1],
        'Baseline_Prop': p_baseline,
        'Baseline_CI_Low': ci_baseline[0],
        'Baseline_CI_High': ci_baseline[1],
        'Odds_Ratio': odds_ratio
    }

# --- Main Analysis Function -----------------------------------------------
def run_analysis():
    try:
        df = pd.read_csv(FILEPATH, header=[0, 1])
    except FileNotFoundError:
        print(f"Error: File not found at {FILEPATH}")
        return
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    # Set index
    if ('Bug ID', 'Unnamed: 0_level_1') in df.columns:
        df = df.set_index(('Bug ID', 'Unnamed: 0_level_1'))
    elif 'Bug ID' in df.columns:
        df = df.set_index('Bug ID')
    else:
        print("Warning: 'Bug ID' column not found.")

    # Convert 'Y' to True, everything else to False
    df = df.applymap(lambda x: True if str(x).strip() == 'Y' else False)

    if PRIMARY_MODEL_TUPLE not in df.columns:
        print(f"Error: Primary model column '{PRIMARY_MODEL_TUPLE}' not found.")
        return

    primary_col = df[PRIMARY_MODEL_TUPLE]
    baseline_cols = [col for col in df.columns if col != PRIMARY_MODEL_TUPLE]
    n_bugs = len(primary_col)

    print(f"--- McNemar's Test: {PRIMARY_MODEL_TUPLE} vs. All Baselines ---")
    print(f"Total Bugs (N) = {n_bugs}\n")
    print("-" * 120)

    results: List[Dict[str, Any]] = []
    for col_name in baseline_cols:
        stats = calculate_mcnemar_stats(primary_col, df[col_name])
        stats['Comparison'] = f"{col_name[0]}_{col_name[1]}"
        results.append(stats)

    if not results:
        print("No comparisons were run.")
        return

    results_df = pd.DataFrame(results)

    # --- Columns for display ---
    cols_of_interest = [
        'Comparison',
        'p_value',
        'Prop_Diff',
        'Prop_Diff_CI_Low',
        'Prop_Diff_CI_High',
        'RepGen_Prop',
        'RepGen_CI_Low',
        'RepGen_CI_High',
        'Baseline_Prop',
        'Baseline_CI_Low',
        'Baseline_CI_High',
        'Unique_Bugs_Primary (b)',
        'Baseline (c)',
        'Both_Reproduced (a)',
        'None_Reproduced (d)',
        'Odds_Ratio'
    ]

    results_df_display = results_df[[c for c in cols_of_interest if c in results_df.columns]]

    # --- Multiple testing correction ---
    reject, p_adjusted, _, _ = multipletests(results_df_display['p_value'], alpha=0.05, method='fdr_bh')
    results_df_display['q_value (adj p-val)'] = p_adjusted
    results_df_display['Significant (α=0.05)'] = reject

    # Reorder columns
    final_cols = [
        'Comparison', 'q_value (adj p-val)', 'Significant (α=0.05)', 'p_value',
        'Prop_Diff', 'Prop_Diff_CI_Low', 'Prop_Diff_CI_High',
        'RepGen_Prop', 'RepGen_CI_Low', 'RepGen_CI_High',
        'Baseline_Prop', 'Baseline_CI_Low', 'Baseline_CI_High',
        'Unique_Bugs_Primary (b)', 'Baseline (c)', 'Both_Reproduced (a)', 'None_Reproduced (d)',
        'Odds_Ratio'
    ]
    results_df_display = results_df_display[final_cols]

    # Display options
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', 180)

    # Print results
    print(results_df_display.to_string(index=False))


if __name__ == "__main__":
    run_analysis()

--- McNemar's Test: ('RepGen', 'Baseline') vs. All Baselines ---
Total Bugs (N) = 106

------------------------------------------------------------------------------------------------------------------------
                Comparison  q_value (adj p-val)  Significant (α=0.05)      p_value  Prop_Diff  Prop_Diff_CI_Low  Prop_Diff_CI_High  RepGen_Prop  RepGen_CI_Low  RepGen_CI_High  Baseline_Prop  Baseline_CI_Low  Baseline_CI_High  Unique_Bugs_Primary (b)  Baseline (c)  Both_Reproduced (a)  None_Reproduced (d)  Odds_Ratio
      Llama-3-8B_Zero-Shot         1.953814e-15                  True 2.442267e-16   0.613208          0.506055           0.659049     0.801887       0.716047        0.866611       0.188679         0.125593          0.273541                       69             4                   16                   17   15.444444
       Llama-3-8B_Few-Shot         1.309396e-10                  True 7.638141e-11   0.471698          0.351413           0.538565     0.801887       0.7160

## Developer Study Results - Statistical Significance Tests

In [11]:
import pandas as pd

# Load and merge data
control = pd.read_csv("ControlGroup.csv")
experimental = pd.read_csv("ExperimentalGroup.csv")
control["Group"] = 0
experimental["Group"] = 1
data = pd.concat([control, experimental])

# Define success (Q9 for TensorFlow, Q13 for PyTorch)
data["Success_TF"] = (data["Q8"] == "Yes").astype(int)
data["Success_PT"] = (data["Q12"] == "Yes").astype(int)

group_success = data.groupby("Group").agg({"Success_TF": "sum", "Success_PT": "sum", "RespondentId": "count"})
group_success.columns = ["TF_Success", "PT_Success", "Total_Bugs"]

# Total successes per group
group_success["Total_Success"] = group_success["TF_Success"] + group_success["PT_Success"]
group_success["Total_Bugs"] = group_success["Total_Bugs"] * 2
print(group_success)

       TF_Success  PT_Success  Total_Bugs  Total_Success
Group                                                   
0              10           9          26             19
1              14          13          28             27


In [12]:
from scipy.stats import chi2_contingency

contingency_table = [
    [group_success.loc[0, "Total_Bugs"] - group_success.loc[0, "Total_Success"],
     group_success.loc[0, "Total_Success"]],
    [group_success.loc[1, "Total_Bugs"] - group_success.loc[1, "Total_Success"],
     group_success.loc[1, "Total_Success"]]
]

chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared p-value: {p:.4f}")

Chi-squared p-value: 0.0423


In [13]:
import numpy as np

# Calculate Phi coefficient for 2x2 table
phi = np.sqrt(chi2 / group_success["Total_Bugs"].sum())
print(f"Phi coefficient: {phi:.3f}")

Phi coefficient: 0.276


In [14]:
import pandas as pd

# Load data
control = pd.read_csv("ControlGroup.csv")
experimental = pd.read_csv("ExperimentalGroup.csv")

# Label groups
control["Group"] = 0  # Control
experimental["Group"] = 1  # Experimental
data = pd.concat([control, experimental])

# Extract time taken (in minutes)
data["Time_TF"] = data["Q9"].astype(float)  # TensorFlow bug time
data["Time_PT"] = data["Q13"].astype(float)  # PyTorch bug time

# Drop rows with missing time values
data = data.dropna(subset=["Time_TF", "Time_PT"])

In [15]:
# Aggregate time for all bugs (TF + PT)
time_data = pd.concat([
    data[["Group", "Time_TF"]].rename(columns={"Time_TF": "Time"}),
    data[["Group", "Time_PT"]].rename(columns={"Time_PT": "Time"})
])

# Group statistics
group_stats = time_data.groupby("Group")["Time"].agg(["mean", "median", "std", "count"])
print(group_stats)

            mean  median       std  count
Group                                    
0      25.730769    25.0  9.681147     26
1      11.107143    10.5  5.079656     28


In [16]:
from scipy.stats import mannwhitneyu

control_time = time_data[time_data["Group"] == 0]["Time"]
experimental_time = time_data[time_data["Group"] == 1]["Time"]

stat, p = mannwhitneyu(control_time, experimental_time, alternative="greater")
print(f"Mann-Whitney U p-value: {p:.4f}")

Mann-Whitney U p-value: 0.0000


In [17]:
import numpy as np

def cliffs_delta(x, y):
    """
    Calculate Cliff's delta effect size.
    """
    n_x = len(x)
    n_y = len(y)
    greater = sum(1 for xi in x for yi in y if xi > yi)
    less = sum(1 for xi in x for yi in y if xi < yi)
    delta = (greater - less) / (n_x * n_y)
    return delta

d = cliffs_delta(control_time, experimental_time)
print(f"Cliff's delta: {abs(d):.2f}")

Cliff's delta: 0.84
